<a href="https://colab.research.google.com/github/amathie5/PPS-Project-/blob/main/PPS_Project_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## Parameters
horizon = 12    #in months
initial_workers = 90        # Initial number of workers
production_per_worker_regular = 10          #Production rate per worker per month
starting_inventory = 900     # Initial inventory
desired_ending_inventory = 1000    # Final inventory target by end of December
inventory_cost_per_unit = 1000     # Inventory holding cost per watch per month (security/insurance)
wage_per_worker = 7000   # Regular wage/worker/month
hire_cost = 50000  # Hiring cost per worker
layoff_cost = 25000  # layoff cost per worker
overtime_percentage = 0.2  # overtime allowance......
overtime_wage_multiplier = 2 # 2x wage_r
subcontracting_cost = 15000 # CHF per watch
max_contracting = 300 # watches per month
overtime_months = {3,5,9,12} # march, may, sept, dec
subcontracting_months = {6,7,10,12} # june, july, oct, dec
demand = {900,950,1200,1050,1100,1300,1250,1100,1300,1450,1500,1700}

In [2]:
def simulate_plan(
    demand,
    production_per_worker_regular,
    starting_inventory,
    desired_ending_inventory,
    initial_workers,
    wage_per_worker,
    hire_cost,
    layoff_cost,
    inventory_cost_per_unit,
    overtime_months,
    overtime_percentage,
    overtime_wage_multiplier,
    strategy,
):
    months = len(demand)
    overtime_months = set(overtime_months or [])
    demand = np.array(demand, dtype=float)

    # workforce arrays
    workers = np.zeros(months)
    hired = np.zeros(months)
    laid_off = np.zeros(months)
    inventory = np.zeros(inventory, dtype=float)
    inventory[0] = starting_inventory

    # helper: overtime factor by month
    ot_flag = np.array([1.0 if t in overtime_months else 0.0 for t in range(months)])
    prod_per_worker = production_per_worker_regular * (1 + ot_flag * overtime_percentage)

    # choose workforce rule
    if strategy == "chase":
        workers_needed = np.ceil(demand / production_per_worker_regular)
    else:
        total_required = demand.sum() + desired_ending_inventory - starting_inventory
        total_capacity_per_worker = prod_per_worker.sum()
        w_const = int(np.ceil(total_required / total_capacity_per_worker))
        workers_needed = np.full(months, w_const)

    # simulate month-by-month
    for t in range(months):
        workers[t] = workers_needed[t]
        if t == 0:
            hired[t] = max(0, workers[t] - initial_workers)
            laid_off[t] = max(0, initial_workers - workers[t])
        else:
            hired[t] = max(0, workers[t] - workers[t-1])
            laid_off[t] = max(0, workers[t-1] - workers[t])
        produced = workers[t] * prod_per_worker[t]
        inventory[t+1] = inventory[t] + produced - demand[t]

    # costs
    wage_cost = workers * wage_per_worker
    hiring_cost = hired * hire_cost
    layoff_costs = laid_off * layoff_cost
    inv_cost = inventory[1:] * inventory_cost_per_unit
    overtime_cost = workers * wage_per_worker * (ot_flag * overtime_percentage) * overtime_wage_multiplier

    total_monthly = wage_cost + hiring_cost + layoff_costs + inv_cost + overtime_cost

    df = pd.DataFrame({
        "Month": np.arange(1, months+1),
        "Workers": workers,
        "Hired": hired,
        "Laid off": laid_off,
        "Produced": workers * prod_per_worker,
        "Demand": demand,
        "End Inventory": inventory[1:],
        "Wage Cost": wage_cost,
        "Overtime Cost": overtime_cost,
        "Hiring Cost": hiring_cost,
        "Layoff Cost": layoff_costs,
        "Inventory Cost": inv_cost,
        "Total Cost": total_monthly,
    })

    summary = {
        "Total Wage": float(wage_cost.sum()),
        "Total Overtime": float(overtime_cost.sum()),
        "Total Hiring": float(hiring_cost.sum()),
        "Total Layoff": float(layoff_costs.sum()),
        "Total Inventory": float(inv_cost.sum()),
        "Grand Total": float(total_monthly.sum()),
    }

    return df, summary